# Lab 3 – Search and A*: Warehouse Robot
Run the cells top to bottom. Sections follow Tasks 0–6 of the lab sheet.

## Task 0 – Search problem formulation
| Component | Specification |
|---|---|
| State S | Robot position `(row, col)` on a non-obstacle cell |
| Actions A | Up, Down, Left, Right |
| Transition T | `T((r,c), a) = (r+dr, c+dc)` if that cell is inside the map and not `#` |
| Initial state s0 | Position of `S` → (1, 1) |
| Goal G | `{ position of 'G' }` → (7, 15) |
| Cost c | 1 per move |

## Task 1 – Design (written before using the LLM)
1. **State**: tuple `(row, col)`.
2. **Warehouse**: list of lists of characters, parsed from the ASCII text.
3. **Valid actions**: for each of 4 directions, the new cell must be inside the grid and not `#`.
4. **Goal test**: `state == goal`.
5. **Frontier**: min-heap of `(f, h, counter, state)`; plus dicts `g` (best cost so far) and `parent`.
6. **Path reconstruction**: follow `parent` pointers from goal back to start, then reverse.
7. **Report**: found?, path, path length, number of states expanded.

In [ ]:
import heapq, math
from collections import deque

WAREHOUSE = """#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################"""

def parse_map(text):
    """ASCII text -> (grid, start, goal)."""
    grid = [list(row) for row in text.strip().splitlines()]
    start = goal = None
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "S": start = (r, c)
            elif ch == "G": goal = (r, c)
    return grid, start, goal

ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

def valid_moves(grid, state):
    """Transition function: yields (action, next_state) for every legal move."""
    r, c = state
    for name, (dr, dc) in ACTIONS.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield name, (nr, nc)

# ---- Heuristics ----
def manhattan(s, g):        return abs(s[0] - g[0]) + abs(s[1] - g[1])
def euclidean(s, g):        return math.hypot(s[0] - g[0], s[1] - g[1])
def zero(s, g):             return 0
def double_manhattan(s, g): return 2 * manhattan(s, g)

def reconstruct(parent, goal):
    path, s = [], goal
    while s is not None:
        path.append(s)
        s = parent[s]
    return path[::-1]

## Task 2 – A* implementation

In [ ]:
def a_star(grid, start, goal, h=manhattan):
    """A* search. Frontier entries: (f, h, tie_counter, state).
    Ties on f are broken by smaller h, which makes A* head toward the goal."""
    counter = 0
    frontier = [(h(start, goal), h(start, goal), counter, start)]
    g = {start: 0}              # best known cost from start
    parent = {start: None}      # for path reconstruction
    closed = set()              # already expanded states
    expanded = 0

    while frontier:
        f, _, _, s = heapq.heappop(frontier)   # lowest f = g + h
        if s in closed:                        # stale duplicate entry
            continue
        if s == goal:                          # goal test
            path = reconstruct(parent, s)
            return dict(found=True, path=path, length=len(path) - 1, expanded=expanded)
        closed.add(s)
        expanded += 1
        for _, nxt in valid_moves(grid, s):
            new_g = g[s] + 1                   # every move costs 1
            if nxt not in g or new_g < g[nxt]:
                g[nxt] = new_g
                parent[nxt] = s
                counter += 1
                hn = h(nxt, goal)
                heapq.heappush(frontier, (new_g + hn, hn, counter, nxt))
    return dict(found=False, path=None, length=None, expanded=expanded)

def bfs(grid, start, goal):
    """Breadth-first search (blind) for comparison."""
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        s = frontier.popleft()
        if s == goal:
            path = reconstruct(parent, s)
            return dict(found=True, path=path, length=len(path) - 1, expanded=expanded)
        expanded += 1
        for _, nxt in valid_moves(grid, s):
            if nxt not in parent:
                parent[nxt] = s
                frontier.append(nxt)
    return dict(found=False, path=None, length=None, expanded=expanded)

def show(grid, path):
    """Draw the path with '*' on the map."""
    out = [row[:] for row in grid]
    for r, c in path[1:-1]:
        out[r][c] = "*"
    print("\n".join("".join(row) for row in out))

## Task 3 – Tests

In [ ]:
# Test 1: original warehouse
grid, start, goal = parse_map(WAREHOUSE)
res = a_star(grid, start, goal)
print("Found:", res["found"], "| Length:", res["length"], "| Expanded:", res["expanded"])
print("Path:", res["path"])
show(grid, res["path"])

In [ ]:
# Test 2: trivial case
g2, s2, t2 = parse_map("#####\n#SG##\n#####")
r = a_star(g2, s2, t2)
assert r["found"] and r["length"] == 1
print("Test 2 passed:", r)

# Test 3: no solution
g3, s3, t3 = parse_map("#######\n#S....#\n###.###\n#...#G#\n#######")
r = a_star(g3, s3, t3)
assert not r["found"]
print("Test 3 passed (reported failure, no infinite loop):", r)

# Test 4: alternative paths -> A* must match BFS (BFS is optimal for unit costs)
g4, s4, t4 = parse_map("#######\n#S....#\n#.###.#\n#....G#\n#######")
ra, rb = a_star(g4, s4, t4), bfs(g4, s4, t4)
assert ra["length"] == rb["length"] == 6
print("Test 4 passed: A* length", ra["length"], "= BFS length", rb["length"])

# Extra: A* length equals BFS length on the warehouse
assert a_star(grid, start, goal)["length"] == bfs(grid, start, goal)["length"]
print("All tests passed")

## Task 4 – Where each concept appears in the code

| Concept | Location |
|---|---|
| State | `(row, col)` tuples, e.g. `s`, `nxt` |
| Action | `ACTIONS` dict (Up/Down/Left/Right) |
| Transition | `valid_moves()` |
| Goal test | `if s == goal:` in `a_star` |
| g(n) | `g` dict, `new_g = g[s] + 1` |
| h(n) | `h(nxt, goal)` → `manhattan()` |
| f(n) | `new_g + hn` (first item of the heap tuple) |
| Frontier | `frontier` list used as a heap via `heapq` |
| Visited states | `closed` set (plus `g` dict) |
| Path reconstruction | `parent` dict + `reconstruct()` |

## Task 5 – BFS vs A*

In [ ]:
a, b = a_star(grid, start, goal), bfs(grid, start, goal)
print(f"{'Measure':<16}{'BFS':>8}{'A*':>8}")
print(f"{'Solution found':<16}{str(b['found']):>8}{str(a['found']):>8}")
print(f"{'Path length':<16}{b['length']:>8}{a['length']:>8}")
print(f"{'States expanded':<16}{b['expanded']:>8}{a['expanded']:>8}")
print("Free cells in warehouse:", sum(ch != "#" for row in grid for ch in row))

## Task 6 – Heuristic experiments
The warehouse is a winding maze, so it is a poor test of heuristics. I therefore also use two extra maps:
an **open room** (heuristics matter a lot) and a **trap map** (shows what an over-estimating heuristic does).

In [ ]:
OPEN_ROOM = """############
#S.........#
#....##....#
#....##....#
#..........#
#....#.....#
#....#.....#
#....#.....#
#.........G#
############"""

TRAP = """##########
#S.#.....#
#..#..#.##
#.#....#.#
#..#.#...#
##.......#
#......#G#
##########"""

heuristics = {"h = 0": zero, "Manhattan": manhattan,
              "Euclidean": euclidean, "2 x Manhattan": double_manhattan}

for name, text in [("WAREHOUSE", WAREHOUSE), ("OPEN ROOM", OPEN_ROOM), ("TRAP MAP", TRAP)]:
    gr, s, t = parse_map(text)
    opt = bfs(gr, s, t)["length"]
    print(f"\n{name}  (optimal length = {opt})")
    print(f"{'Heuristic':<16}{'Found':>7}{'Length':>8}{'Expanded':>10}{'Optimal?':>10}")
    for hname, h in heuristics.items():
        r = a_star(gr, s, t, h)
        print(f"{hname:<16}{str(r['found']):>7}{r['length']:>8}{r['expanded']:>10}{str(r['length']==opt):>10}")

## Reading the results
- **h = 0** → A* becomes uniform-cost search (like BFS here): optimal but explores a lot.
- **Manhattan** → admissible and consistent for 4-way moves: optimal and usually fewer expansions.
- **Euclidean** → admissible but weaker (smaller than Manhattan), so more expansions.
- **2 × Manhattan** → *inadmissible*: very few expansions but the path can be longer than optimal (see trap map).